# Native QML: variational energy and NOT training

Write the quantum programs as native `qfunc` functions and run both original
workflow sections locally. VQE retains zero initial angles and at most 200
iterations. NOT retains initial weight 3, MAE, SGD learning rate 1 and five epochs.
Exact local objectives and the existing native Nelder-Mead optimizer are explicit
adaptations; identical remote sampled costs or optimizer trajectories are not claimed.


In [ ]:
from matplotlib import pyplot as plt

from examples.synthesis.tutorials.basics.qml.overview.workflow import (
    Workflow,
    plot_workflow,
    solve_energy,
    train_not,
)
from flagquantum.drawer import draw

variational = solve_energy(maxiter=200)
print("Energy:", variational.optimizer.value)
print("Angles:", variational.optimizer.parameters)
print("Iterations:", variational.optimizer.iterations)
print("Evaluations:", variational.optimizer.evaluations)
print("Converged:", variational.optimizer.converged)


The ansatz uses the original single-qubit three-angle U function. Its cost is
`0.5 I + 0.5 Z - X`, evaluated from actual quantum execution. The final circuit
and actual synthesis resource report remain available for inspection.


In [ ]:
print("VQE resources:", variational.program.report["resources"])
print("VQE clean workspace:", variational.program.report["workspace_qubits"])
print(draw(variational.program.circuit, show_all_qubits=True))


## Train the NOT task

The original recorded data are interpreted as input angles `(pi, 0)` and labels
`(1, 0)` in float64. Native qfunc real parameters are bound at synthesis time.
This teaching workflow explicitly obtains each weight derivative from two shifted
quantum executions and assigns it to the parameter owned by PyTorch SGD.
It does not silently differentiate through a float conversion or provide a generic
QLayer. Each epoch uses the complete two-example batch and the original MAE.


In [ ]:
training = train_not(epochs=5, initial_weight=3.0, learning_rate=1.0)
for epoch, step in enumerate(training.steps):
    print(epoch, step)
print("Final weight:", training.weight)
print("Predictions for pi and zero:", training.predictions)
print("Accuracy at absolute tolerance 1e-2:", training.accuracy)
print("NOT resources:", training.program.report["resources"])


## Inspect the actual cost histories

Each VQE entry is an actual native objective evaluation; each training point is
the MAE before that epoch's update. Resource statistics describe the final bound
programs, not the sum of all shifted evaluation circuits. No source outputs are
used as native results, and no quantum advantage is claimed.


In [ ]:
workflow = Workflow(variational, training)
figure = plot_workflow(workflow)
plt.show()
figure
